In [6]:
athletes = {
    "Athlete12": {
        "sport": "rowing",
        "ftp": 230,
        "fit_dir": "AthleteDataCoding/Athlete12/OGs",
        "label_dir": "AthleteDataCoding/Athlete12/GTs",
        "allowed_files": {
            "10536403349_5006_row.fit",
            "10543962115_10006_row.fit",
            "10551999765_10006_row.fit",
            "10652950510_Btchen_fahren.fit",
            "10674304801_Btchen_fahren_in_Etappen_3.fit",
            "10694767945_Btchen_fahren_.fit",
            "10809067165_3007_row.fit",
            "11636429453_4558_row.fit",
            "11783093951_4x2000_sub8.fit",
            "11791568584_.fit",
            "11808467517_.fit",
            "11838948742_3006_row.fit",
            "11846980624_Platt_.fit",
            "11855866225_In_den_Seilen.fit",
            "11864381887_Catwalk_.fit",
            "11912062341_500er_in_grau.fit",
            "11962243206_Wundmanagement.fit",
            "11971395278_Use_it_or_lose_it.fit",
            "11987690514_Besser_als_Nix.fit",
            "11994450315_Airobics.fit",
            "12036692734_Exhausted.fit",
            "12069656901_Schwitzen_im_Sitzen.fit",
            "12501679452_Zustand_nach_Xtem_Atemwegsinfekt.fit",
            "12806981726_Row_Stretch__Stabi.fit",
            "12846436186_Synchronflug.fit",
            "12927701413_I_have_no_idea_when_Ill_be_back_in_serious_training_routine_but_at_least_we_have_a_kitchen.fit",
            "12951604563_DienstSport.fit",
            "13010348229_1h_w_4x1_intensity.fit",
            "13039020832_Analytiker.fit",
            "13280559542_Warm_up_rowing.fit",
            "13300350440_W_Up.fit",
            "13363035398_SGAktiv.fit",
            "13582048984_W_Up.fit",
            "13583093636_Afterburner_.fit",
            "13601462878_Zehnbauer.fit",
            "13609970768_Uffwrme.fit",
            "13610691264_1x_Crescendo.fit",
            "13618782252_3x5.fit",
            "13643807487_Nachfitten.fit",
            "13662882990_Heldentod.fit",
            "13672121049_Base_Miles.fit",
            "13688068283_Luftpresser.fit",
            "13918354210_W_Upen.fit",
            "13957096402_Technik.fit",
            "13971240869_A_ella_le_gusta.fit",
            "13974345688_Nochmaaaal.fit",
            "13983533934_Technik__30er.fit",
            "14001095362_Wer_will_der_kann_.fit",
            "14038989670__Hyperthermie_.fit",
            "14077735636_Base.fit",
            "14089880174_Zn_IKEA.fit",
            "14114545767_Dampfnudel.fit",
            "14125110656_Vallah_isch_balla.fit",
            "14135321532_Pimp_my_ride.fit",
            "14156450361_On_a_mission.fit",
            "14174927764_Dunstabzugshaubenselfie.fit",
            "14182817844_The_Emptiness_Machine.fit",
            "14260930602_3x1010_SubThr.fit",
            "14313279747_Vernunft_verliert_.fit",
            "14374019349_Uff.fit",
            "14396237986_4659_row.fit"
        },
    },
    "Athlete2": {
        "sport": "biking",
        "ftp": 341,
        "fit_dir": "AthleteDataCoding/Athlete2/OGs",
        "label_dir": "AthleteDataCoding/Athlete2/GTs",
        "allowed_files": "all"
    }
}

In [7]:
import pandas as pd
from garmin_fit_sdk import Decoder, Stream
import numpy as np
import os
from scipy.optimize import minimize
from datetime import timedelta
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

##### All functions for data processing and model processing

In [8]:
def read_fit_file(fit_file_path: str) -> pd.DataFrame:
    """
    Reads a FIT file and returns a raw DataFrame of records without any processing.

    Parameters:
    fit_file_path (str): Path to the FIT file.

    Returns:
    pd.DataFrame: Raw DataFrame extracted from the FIT file.
    """
    stream = Stream.from_file(fit_file_path)
    decoder = Decoder(stream)
    messages, errors = decoder.read()

    if errors:
        print(f"Found {len(errors)} errors in {fit_file_path}")

    records = []
    for record in messages.get("record_mesgs", []):
        record_data = {}
        for field_name, field_value in record.items():
            if isinstance(field_value, dict):
                for subfield, value in field_value.items():
                    record_data[f"{field_name}_{subfield}"] = value
            else:
                record_data[field_name] = field_value
        records.append(record_data)

    df = pd.DataFrame(records)
    return df

In [9]:
def prepare_dataframe(
    df: pd.DataFrame,
    ftp: float,
    window_size: int = 5,
    use_roll_avg: bool = True,
    seven_zone_model: bool = True
) -> pd.DataFrame:
    """
    Prepares a DataFrame by cleaning, interpolating, applying rolling average,
    and classifying power zones.

    Parameters:
    df (pd.DataFrame): Raw DataFrame from FIT file.
    ftp (float): Functional Threshold Power used to compute power zones.
    window_size (int, optional): Rolling window size. Defaults to 5.
    use_roll_avg (bool, optional): Use rolling average for power zone classification. Defaults to True.
    seven_zone_model (bool, optional): Use 7-zone model if True, else 6-zone. Defaults to True.

    Returns:
    pd.DataFrame: Cleaned and enriched DataFrame with timestamps and power zones.
    """
    if df.empty or 'timestamp' not in df or 'power' not in df:
        print("Input DataFrame is empty or lacks required fields.")
        return df

    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df = df.sort_values('timestamp').reset_index(drop=True)

    # Remove duplicate timestamps
    df = df.drop_duplicates(subset='timestamp', keep='first')

    # Reindex to expected 1Hz frequency
    expected = pd.date_range(start=df['timestamp'].min(), end=df['timestamp'].max(), freq='1s')
    df = df.set_index('timestamp').reindex(expected).reset_index().rename(columns={'index': 'timestamp'})

    # Apply rolling average to power
    df['power_roll_avg'] = df['power'].rolling(window_size, center=True, min_periods=1).mean()

    # Select power source for classification
    df['power_for_classification'] = df['power_roll_avg'] if use_roll_avg else df['power']

    # Define power zones
    if seven_zone_model:
        power_bins = [0, 55, 75, 90, 105, 120, 150, float('inf')]
        power_labels = ['Zone1', 'Zone2', 'Zone3', 'Zone4', 'Zone5', 'Zone6', 'Zone7']
    else:
        power_bins = [0, 55, 75, 90, 105, 120, float('inf')]
        power_labels = ['Zone1', 'Zone2', 'Zone3', 'Zone4', 'Zone5', 'Zone6']

    df['Power_Zone'] = pd.cut(df['power_for_classification'] / ftp * 100, bins=power_bins, labels=power_labels)
    df['Interval_Type'] = df['Power_Zone'].astype(str)

    return df

In [10]:
def apply_watt_drop_detection(
    df: pd.DataFrame,
    watt_drop: float = 20,
    extend_gradient: bool = True
) -> pd.DataFrame:
    """
    Detects rapid increases and decreases in power (watt_drop threshold) to mark
    potential interval starts and ends. Optionally extends detection based on gradients.

    Parameters:
    df (pd.DataFrame): Input DataFrame with timestamp and power_roll_avg columns.
    watt_drop (float, optional): Threshold in watts/second to detect interval transitions. Defaults to 20.
    extend_gradient (bool, optional): Whether to extend detection using average gradient. Defaults to True.

    Returns:
    pd.DataFrame: Updated DataFrame with interval detection columns added.
    """

    df = df.copy()

    df['delta_time'] = df['timestamp'].diff().dt.total_seconds()
    df['power_derivative'] = df['power_roll_avg'].diff() / df['delta_time']

    df['interval_start_candidate'] = (df['power_derivative'] > watt_drop).fillna(False)
    df['interval_end_candidate'] = (df['power_derivative'] < -watt_drop).fillna(False)

    # Confirmed start
    cand = df['interval_start_candidate']
    df['confirmed_interval_start'] = (
        cand & cand.shift(-1, fill_value=False)) | \
        (cand.shift(1, fill_value=False) & cand) | \
        (cand.shift(2, fill_value=False) & cand.shift(1, fill_value=False)) | \
        (cand & cand.shift(-2, fill_value=False))

    # Confirmed end
    cand = df['interval_end_candidate']
    df['confirmed_interval_end'] = (
        cand & cand.shift(-1, fill_value=False)) | \
        (cand.shift(1, fill_value=False) & cand) | \
        (cand.shift(2, fill_value=False) & cand.shift(1, fill_value=False)) | \
        (cand & cand.shift(-2, fill_value=False))

    if extend_gradient:
        df['extended_start'] = False
        in_interval = False
        for i in df.index[::-1]:
            if df.at[i, 'interval_start_candidate']:
                in_interval = True
            elif in_interval:
                t = df.at[i, 'timestamp']
                segment = df[(df['timestamp'] >= t - timedelta(seconds=5)) & (df['timestamp'] < t)]
                if not segment.empty and segment['power_derivative'].mean() > 0:
                    df.at[i, 'extended_start'] = True
                else:
                    in_interval = False

        df['extended_end'] = False
        in_interval = False
        for i in df.index:
            if df.at[i, 'interval_end_candidate']:
                in_interval = True
            elif in_interval:
                t = df.at[i, 'timestamp']
                segment = df[(df['timestamp'] > t) & (df['timestamp'] <= t + timedelta(seconds=10))]
                if not segment.empty and segment['power_derivative'].mean() < 0:
                    df.at[i, 'extended_end'] = True
                else:
                    in_interval = False

        df['interval_start_candidate'] |= df['extended_start']
        df['interval_end_candidate'] |= df['extended_end']
        df.drop(columns=['extended_start', 'extended_end'], inplace=True)

    return df


def group_true_values(
    df: pd.DataFrame,
    start_column: str = 'interval_start_candidate',
    end_column: str = 'interval_end_candidate'
) -> pd.DataFrame:
    """
    Identifies the first and last rows of each group of consecutive True values
    in boolean flag columns, typically marking interval starts and ends.

    Parameters:
    df (pd.DataFrame): DataFrame containing boolean flag columns.
    start_column (str): Name of the start candidate column. Defaults to 'interval_start_candidate'.
    end_column (str): Name of the end candidate column. Defaults to 'interval_end_candidate'.

    Returns:
    pd.DataFrame: DataFrame with 'group_first' and 'group_last' columns added.
    """

    # Shifted versions for neighborhood check
    prev1_end = df[start_column].shift(1, fill_value=False)
    next1_end = df[end_column].shift(-1, fill_value=False)
    prev2_end = df[start_column].shift(2, fill_value=False)
    next2_end = df[end_column].shift(-2, fill_value=False)

    # Identify groups of True values in the sequence
    group_start = (df[start_column] == True) & (prev1_end == False) & (prev2_end == False)
    group_end = (df[end_column] == True) & (next1_end == False) & (next2_end == False)

    # Only keep the first occurrence of a group as True
    df['group_first'] = group_start
    df['group_last'] = group_end

    return df

def enforce_consecutive_intervals(
    df: pd.DataFrame,
    start_col: str = 'group_first',
    end_col: str = 'group_last'
) -> pd.DataFrame:
    """
    Ensures that interval starts and ends alternate properly and that the first and last
    rows in the DataFrame are explicitly marked as start and end points.

    Parameters:
    df (pd.DataFrame): DataFrame with group_first and group_last boolean columns.
    start_col (str): Column name for start flags. Defaults to 'group_first'.
    end_col (str): Column name for end flags. Defaults to 'group_last'.

    Returns:
    pd.DataFrame: Updated DataFrame with enforced consecutive interval boundaries.
    """

    df = df.copy()

    # Start: if previous row is a group_end → current row should be a group_start
    prev_end = df[end_col].shift(1, fill_value=False)
    df.loc[prev_end & (~df[start_col]), start_col] = True

    # End: if next row is a group_start → current row should be a group_end
    next_start = df[start_col].shift(-1, fill_value=False)
    df.loc[next_start & (~df[end_col]), end_col] = True

    # Force the very first row to be a start
    df.loc[df.index[0], start_col] = True

    # Force the very last row to be an end
    df.loc[df.index[-1], end_col] = True

    return df

def assign_dominant_zone_type_per_interval(
    df: pd.DataFrame,
    start_col: str = 'group_first',
    label_col: str = 'Interval_Type'
) -> pd.DataFrame:
    """
    Assigns the dominant zone label to each interval group using the mode of the values
    in the label column (e.g., power zone classification).

    Parameters:
    df (pd.DataFrame): Input DataFrame with intervals and zone labels.
    start_col (str): Column marking interval starts. Defaults to 'group_first'.
    label_col (str): Column containing zone labels. Defaults to 'Interval_Type'.

    Returns:
    pd.DataFrame: DataFrame with an added 'interval_zone_type' column.
    """

    df = df.copy()
    df[label_col] = df[label_col].replace({np.nan: 'Unclassified', 'nan': 'Unclassified'})

    # Assign a group ID to each interval based on cumulative sum of start flags
    df['interval_group_id'] = df[start_col].cumsum()

    # Find the most frequent zone in each group
    dominant_zones = (
        df.groupby('interval_group_id')[label_col]
        .agg(lambda x: x.value_counts().idxmax())
        .rename('interval_zone_type')
    )

    # Merge the result back to the original DataFrame
    df = df.merge(dominant_zones, left_on='interval_group_id', right_index=True)

    return df

def merge_consecutive_same_zone_intervals(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Merges adjacent intervals with the same zone type and marks the first and last rows
    of each merged segment using 'final_group_start' and 'final_group_end'.

    Parameters:
    df (pd.DataFrame): DataFrame containing 'interval_zone_type' and 'timestamp'.

    Returns:
    pd.DataFrame: DataFrame with merged intervals and updated group markings.
    """

    df = df.copy()
    df = df.sort_values(by='timestamp').reset_index(drop=True)

    # Detect zone changes from row to row
    zone_change = (df['interval_zone_type'] != df['interval_zone_type'].shift()).astype(int)

    # New group ID is just cumulative sum of changes
    df['interval_group_id'] = zone_change.cumsum()

    # Re-assign interval_zone_type per new group (ensures consistent label)
    new_zones = (
        df.groupby('interval_group_id')['interval_zone_type']
        .agg(lambda x: x.value_counts().idxmax())
    )
    df['interval_zone_type'] = df['interval_group_id'].map(new_zones)

    # Initialize new columns
    df['final_group_start'] = False
    df['final_group_end'] = False

    # Mark first and last index of each group
    group_indices = df.groupby('interval_group_id').agg(first_idx=('timestamp', 'idxmin'),
                                                        last_idx=('timestamp', 'idxmax'))

    df.loc[group_indices['first_idx'], 'final_group_start'] = True
    df.loc[group_indices['last_idx'], 'final_group_end'] = True

    return df

def detect_and_invalidate_stop_resume_events(
    df: pd.DataFrame,
    window_seconds: int = 20,
    drop_threshold: float = 100,
    recovery_margin: float = 10
) -> pd.DataFrame:
    """
    Detects and invalidates stop-resume events characterized by sudden drops and recoveries
    in power. Removes interval markers and fills with previous group/zone metadata.

    Parameters:
    df (pd.DataFrame): DataFrame with power_roll_avg, final_group_start/end, etc.
    window_seconds (int): Size of the sliding window in seconds. Defaults to 20.
    drop_threshold (float): Minimum power drop to flag stop event. Defaults to 100.
    recovery_margin (float): Margin for recovery (return to baseline). Defaults to 10.

    Returns:
    pd.DataFrame: Cleaned DataFrame with short stop-resume interruptions removed.
    """

    df = df.copy()
    df = df.sort_values("timestamp").reset_index(drop=True)
    df = df.dropna(subset=["power_roll_avg"])

    power = df["power_roll_avg"].values
    timestamps = df["timestamp"].values

    for i in range(len(df) - window_seconds):
        window_power = power[i:i + window_seconds]
        window_time = timestamps[i:i + window_seconds]

        pre_value = window_power[0]
        min_idx = window_power.argmin()
        min_value = window_power[min_idx]
        post_value = window_power[-1]

        drop = pre_value - min_value
        recovered = abs(post_value - pre_value) <= recovery_margin

        if drop >= drop_threshold and recovered and 1 < min_idx < window_seconds - 2:
            start = window_time[0]  # already UTC-aware
            end = window_time[-1]

            mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)

            # Get last known interval metadata before the drop
            prior = df[df["timestamp"] < start]
            fill_zone = prior["interval_zone_type"].dropna().iloc[-1] if not prior["interval_zone_type"].dropna().empty else None
            fill_group = prior["interval_group_id"].dropna().iloc[-1] if not prior["interval_group_id"].dropna().empty else None

            df.loc[mask, "final_group_start"] = False
            df.loc[mask, "final_group_end"] = False
            df.loc[mask, "interval_group_id"] = fill_group
            df.loc[mask, "interval_zone_type"] = fill_zone

    return df


def merge_short_intervals(
    df: pd.DataFrame,
    min_duration_seconds: int = 5
) -> pd.DataFrame:
    """
    Reassigns short intervals to neighboring ones based on similarity in average
    power to eliminate unstable short segments.

    Parameters:
    df (pd.DataFrame): DataFrame with 'interval_group_id', 'interval_zone_type', 'power_roll_avg'.
    min_duration_seconds (int): Minimum duration (in seconds) to consider an interval valid.

    Returns:
    pd.DataFrame: DataFrame with short intervals merged into neighbors.
    """

    df = df.copy()
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Ensure group ID is consistent and not NaN
    df["interval_group_id"] = df["interval_group_id"].ffill()

    grouped = df.groupby("interval_group_id")

    for group_id, group_df in grouped:
        duration = (group_df["timestamp"].iloc[-1] - group_df["timestamp"].iloc[0]).total_seconds()
        if duration >= min_duration_seconds:
            continue  # skip long enough intervals

        # Mean power of this short interval
        this_mask = (df["timestamp"] >= group_df["timestamp"].iloc[0]) & (df["timestamp"] <= group_df["timestamp"].iloc[-1])
        this_mean = group_df["power_roll_avg"].mean()

        # 5 seconds before
        before_mask = (df["timestamp"] < group_df["timestamp"].iloc[0]) & \
                      (df["timestamp"] >= group_df["timestamp"].iloc[0] - pd.Timedelta(seconds=5))
        before_mean = df.loc[before_mask, "power_roll_avg"].mean()
        before_group = df.loc[before_mask, "interval_group_id"].dropna().iloc[-1] if not df.loc[before_mask].empty else None
        before_zone = df.loc[before_mask, "interval_zone_type"].dropna().iloc[-1] if not df.loc[before_mask].empty else None

        # 5 seconds after
        after_mask = (df["timestamp"] > group_df["timestamp"].iloc[-1]) & \
                     (df["timestamp"] <= group_df["timestamp"].iloc[-1] + pd.Timedelta(seconds=5))
        after_mean = df.loc[after_mask, "power_roll_avg"].mean()
        after_group = df.loc[after_mask, "interval_group_id"].dropna().iloc[0] if not df.loc[after_mask].empty else None
        after_zone = df.loc[after_mask, "interval_zone_type"].dropna().iloc[0] if not df.loc[after_mask].empty else None

        # Choose the closest in mean power
        if pd.isna(before_mean) and pd.isna(after_mean):
            continue  # nowhere to merge to

        if pd.isna(before_mean):
            assign_group, assign_zone = after_group, after_zone
        elif pd.isna(after_mean):
            assign_group, assign_zone = before_group, before_zone
        else:
            if abs(this_mean - before_mean) <= abs(this_mean - after_mean):
                assign_group, assign_zone = before_group, before_zone
            else:
                assign_group, assign_zone = after_group, after_zone

        # Apply reassignment
        df.loc[this_mask, "interval_group_id"] = assign_group
        df.loc[this_mask, "interval_zone_type"] = assign_zone
        df.loc[this_mask, "final_group_start"] = False
        df.loc[this_mask, "final_group_end"] = False

    return df

def reassign_first_n_seconds(
    df: pd.DataFrame,
    seconds: int = 60
) -> pd.DataFrame:
    """
    Assigns the first `n` seconds of data to the first valid interval zone found
    after that period, eliminating early 'Unclassified' segments.

    Parameters:
    df (pd.DataFrame): DataFrame with timestamp, interval_zone_type, final_group_start.
    seconds (int): Number of initial seconds to overwrite with a valid zone.

    Returns:
    pd.DataFrame: DataFrame with reassigned initial interval zone.
    """

    if df.empty:
        return df

    start_time = df["timestamp"].iloc[0]
    cutoff_time = start_time + pd.Timedelta(seconds=seconds)

    # Find first valid zone after cutoff
    future_mask = df["timestamp"] > cutoff_time
    valid_future = df.loc[future_mask & df["interval_zone_type"].notna()]
    valid_future = valid_future[valid_future["interval_zone_type"] != "Unclassified"]

    if valid_future.empty:
        return df  # nothing to assign from

    new_zone = valid_future.iloc[0]["interval_zone_type"]

    # Apply reassignment
    df.loc[df["timestamp"] <= cutoff_time, "interval_zone_type"] = new_zone
    df.loc[valid_future.index[0], "final_group_start"] = False
    df['interval_group_id'] = df['final_group_start'].cumsum()

    return df

##### All functions for extracting the ground truth data and evaluation

In [11]:
# ==== 1. Erkannte Startzeiten extrahieren ====
def extract_detected_starts(df: pd.DataFrame) -> list:
    """
    Extracts all timestamps marked as detected interval starts.

    Parameters:
    df (pd.DataFrame): DataFrame containing a 'final_group_start' boolean column and 'timestamp'.

    Returns:
    list: List of timestamps where 'final_group_start' is True.
    """
    return df[df["final_group_start"]]["timestamp"].tolist()


# ==== 2. Ground-Truth-Startzeiten laden ====
def load_ground_truth_starts(path: str) -> list[dict]:
    """
    Loads ground truth start times from a CSV file and assigns interval zone labels
    based on the mode of 'Interval_Type' in each segment.

    Parameters:
    path (str): Path to the CSV label file containing 'timestamp' and 'Manual_Timestamps'.

    Returns:
    list of dict: Each dict contains 'start', 'end', and 'zone_type' for an interval.
    """
    df = pd.read_csv(path)
    if df.empty or "timestamp" not in df.columns:
        raise ValueError(f"Leeres oder ungültiges Label-File: {path}")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.tz_localize(None)
    df = df.dropna(subset=["timestamp"])

    df["is_start"] = df["Manual_Timestamps"] == True
    starts = df.loc[df["is_start"], "timestamp"].tolist()

    if not starts or df["timestamp"].iloc[0] not in starts:
        starts.insert(0, df["timestamp"].iloc[0])

    ends = starts[1:] + [df["timestamp"].iloc[-1]]

    intervals = []
    for start, end in zip(starts, ends):
        interval_df = df[(df["timestamp"] >= start) & (df["timestamp"] < end)]
        if not interval_df.empty and "Interval_Type" in interval_df.columns:
            zone_mode = interval_df["Interval_Type"].mode()
            zone_type = zone_mode.iloc[0] if not zone_mode.empty else "Unclassified"
        else:
            zone_type = "Unclassified"

        intervals.append({
            "start": start,
            "end": end,
            "zone_type": zone_type
        })

    return intervals

# ==== 3. Ground Truth Dateien für alle Athleten vorbereiten ====
def preload_ground_truths(
    fit_dir: str,
    label_dir: str,
    allowed_files: str | list = "all"
) -> dict:
    """
    Loads ground truth interval metadata for a batch of athletes.

    Parameters:
    fit_dir (str): Directory containing .fit files.
    label_dir (str): Directory containing label CSVs.
    allowed_files (str | list): Either "all" or a list of specific .fit filenames to load.

    Returns:
    dict: Mapping of filename to list of ground truth intervals.
    """
    gt_data = {}
    all_fit_files = [f for f in os.listdir(fit_dir) if f.endswith(".fit")]

    files_to_use = all_fit_files if allowed_files == "all" else [f for f in all_fit_files if f in allowed_files]

    for filename in files_to_use:
        label_path = os.path.join(label_dir, filename.replace(".fit", "_with_manual_labels.csv"))
        if not os.path.exists(label_path):
            continue
        try:
            gt_data[filename] = load_ground_truth_starts(label_path)
        except Exception as e:
            print(f"❌ Fehler beim Laden der Labels für {filename}: {e}")

    return gt_data

# === Weighted F-beta helper ===
def weighted_fbeta(
    precision: float,
    recall: float,
    beta: float = 1.5
) -> float:
    """
    Computes the weighted F-beta score.

    Parameters:
    precision (float): Precision value.
    recall (float): Recall value.
    beta (float): Weighting factor for recall. Defaults to 1.5.

    Returns:
    float: Weighted F-beta score.
    """
    if precision + recall == 0:
        return 0
    return (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall)


# ==== 3. Vergleich mit Toleranz ±5 Sekunden ====
def compare_start_times(
    predicted: list,
    ground_truth: list,
    tolerance_sec: int = 5,
    beta: float = 1.5
) -> tuple:
    """
    Compares predicted vs. ground truth start times with a tolerance window.

    Parameters:
    predicted (list): List of predicted start timestamps.
    ground_truth (list): List of actual (ground truth) start timestamps.
    tolerance_sec (int): Allowed margin of error in seconds. Defaults to 5.
    beta (float): Beta value for F-beta score. Defaults to 1.5.

    Returns:
    tuple: (precision, recall, fbeta_score)
    """
    tolerance = timedelta(seconds=tolerance_sec)
    tp = 0
    matched_gt = set()

    for pred_time in predicted:
        for i, gt_time in enumerate(ground_truth):
            if i in matched_gt:
                continue
            if abs(pred_time - gt_time) <= tolerance:
                tp += 1
                matched_gt.add(i)
                break

    fp = len(predicted) - tp
    fn = len(ground_truth) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    fbeta     = weighted_fbeta(precision, recall, beta=beta)

    return precision, recall, fbeta

In [12]:
# === Expand "allowed_files": "all" ===
for athlete, config in athletes.items():
    if config["allowed_files"] == "all":
        fit_dir = config["fit_dir"]
        try:
            config["allowed_files"] = {
                f for f in os.listdir(fit_dir) if f.endswith(".fit")
            }
        except FileNotFoundError:
            print(f"⚠️ Fit directory not found for {athlete}: {fit_dir}")
            config["allowed_files"] = set()

# === Test Sessions ===
test_sessions = [
    ("Athlete2", "13363782092_Zwift__Aerobic_Mixup_in_New_York"),
    ("Athlete2", "i65696340_Zwift__LC16_Lactate_Clearance_Over_Under_in_Watopia"),
    ("Athlete12", "13601462878_Zehnbauer"),
    ("Athlete12", "12036692734_Exhausted"),
    ("Athlete12", "12846436186_Synchronflug"),
    ("Athlete12", "11962243206_Wundmanagement"),
    ("Athlete12", "13688068283_Luftpresser"),
    ("Athlete12", "11783093951_4x2000_sub8"),
    ("Athlete12", "13983533934_Technik__30er"),
    ("Athlete12", "11846980624_Platt_"),
    ("Athlete12", "14125110656_Vallah_isch_balla"),
]

test_file_set = {name + ".fit" for _, name in test_sessions}

# === Helper to Convert Sessions ===
def get_session_tuples(sessions, exclude_files=False):
    files = []
    for athlete, session in sessions:
        config = athletes.get(athlete)
        if not config:
            print(f"⚠️ Athlete '{athlete}' not found.")
            continue
        fname = session + ".fit"
        allowed = config["allowed_files"]
        if fname not in allowed:
            print(f"⚠️ {fname} not in allowed files for {athlete}")
            continue
        if exclude_files and fname in exclude_files:
            continue
        files.append((fname, config["ftp"], config["fit_dir"], config["label_dir"]))
    return files

# === Collect Test Files ===
test_files = get_session_tuples(test_sessions)
print(f"✅ Using {len(test_files)} manually selected sessions.")

# === Collect Training Files ===
train_files = []
for athlete, config in athletes.items():
    sessions = [(athlete, f[:-4]) for f in config["allowed_files"]]
    train_files.extend(get_session_tuples(sessions, exclude_files=test_file_set))

print(f"✅ Using {len(train_files)} training sessions.")

✅ Using 11 manually selected sessions.
✅ Using 60 training sessions.


##### Optimization

In [13]:
# === Optimization Setup ===
best_score = -1
best_watt_drop = None
best_precision = None
best_recall = None

cached_preprocessed_dfs = {}

for filename, ftp, fit_dir, label_dir in train_files:
    full_path = os.path.join(fit_dir, filename)
    label_path = os.path.join(label_dir, filename.replace(".fit", "_with_manual_labels.csv"))

    if not os.path.exists(label_path):
        print(f"⚠️ Skipping {filename} (no ground truth found)")
        continue

    try:
        df = read_fit_file(full_path)
        df = prepare_dataframe(df, ftp)
        gt_intervals = load_ground_truth_starts(label_path)

        if not gt_intervals:
            print(f"⚠️ Ground truth empty for {filename} → {label_path}")
        else:
            print(f"✅ Loaded ground truth for: {filename} ({len(gt_intervals)} intervals)")

        cached_preprocessed_dfs[filename] = (df, ftp, gt_intervals)
    except Exception as e:
        print(f"❌ Error processing {filename}: {e}")


for wd in range(10, 14):  # test watt_drop values
    precs, recs = [], []

    for filename, (df_base, ftp, gt_intervals) in cached_preprocessed_dfs.items():
        try:
            df = apply_watt_drop_detection(df_base, watt_drop=wd)
            df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)

            df = group_true_values(df)
            df = enforce_consecutive_intervals(df)
            df = assign_dominant_zone_type_per_interval(df)
            df = merge_consecutive_same_zone_intervals(df)
            df = detect_and_invalidate_stop_resume_events(df)
            df = merge_short_intervals(df)
            df = reassign_first_n_seconds(df, seconds=60)

            pred = extract_detected_starts(df)
            gt_starts = [x["start"] for x in gt_intervals]

            precision, recall, _ = compare_start_times(pred, gt_starts)
            precs.append(precision)
            recs.append(recall)

        except Exception as e:
            print(f"⚠️ Error evaluating {filename}: {e}")

    mean_p = np.mean(precs) if precs else 0
    mean_r = np.mean(recs) if recs else 0
    fbeta = weighted_fbeta(mean_p, mean_r)

    if fbeta > best_score:
        best_score = fbeta
        best_watt_drop = wd
        best_precision = mean_p
        best_recall = mean_r

print("\n✅ Beste Kombination:")
print(f"• watt_drop     = {best_watt_drop}")
print(f"• Precision     = {best_precision:.4f}")
print(f"• Recall        = {best_recall:.4f}")
print(f"• Fβ-Score      = {best_score:.4f}")


✅ Loaded ground truth for: 13610691264_1x_Crescendo.fit (6 intervals)
✅ Loaded ground truth for: 10674304801_Btchen_fahren_in_Etappen_3.fit (4 intervals)
✅ Loaded ground truth for: 14374019349_Uff.fit (5 intervals)
✅ Loaded ground truth for: 11987690514_Besser_als_Nix.fit (11 intervals)
✅ Loaded ground truth for: 11971395278_Use_it_or_lose_it.fit (15 intervals)
✅ Loaded ground truth for: 14135321532_Pimp_my_ride.fit (11 intervals)
✅ Loaded ground truth for: 11912062341_500er_in_grau.fit (15 intervals)
✅ Loaded ground truth for: 10652950510_Btchen_fahren.fit (11 intervals)
✅ Loaded ground truth for: 11636429453_4558_row.fit (5 intervals)
✅ Loaded ground truth for: 13300350440_W_Up.fit (6 intervals)
✅ Loaded ground truth for: 14174927764_Dunstabzugshaubenselfie.fit (12 intervals)
✅ Loaded ground truth for: 14396237986_4659_row.fit (9 intervals)
✅ Loaded ground truth for: 11838948742_3006_row.fit (5 intervals)
✅ Loaded ground truth for: 12927701413_I_have_no_idea_when_Ill_be_back_in_serio

##### Evaluation on test data set

In [14]:
def evaluate_on_test(watt_drop, test_files):
    precs, recs = [], []

    for filename, ftp, fit_dir, label_dir in test_files:
        full_path = os.path.join(fit_dir, filename)
        label_path = os.path.join(label_dir, filename.replace(".fit", "_with_manual_labels.csv"))

        if not os.path.exists(label_path):
            print(f"⚠️ Skipping {filename} (no ground truth found)")
            continue

        try:
            df = read_fit_file(full_path, ftp)
            df = prepare_dataframe(df, ftp)
            gt_intervals = load_ground_truth_starts(label_path)

            df = apply_watt_drop_detection(df, watt_drop=watt_drop)
            df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)

            df = group_true_values(df)
            df = enforce_consecutive_intervals(df)
            df = assign_dominant_zone_type_per_interval(df)
            df = merge_consecutive_same_zone_intervals(df)
            df = detect_and_invalidate_stop_resume_events(df)
            df = merge_short_intervals(df)
            df = reassign_first_n_seconds(df, seconds=60)

            pred = extract_detected_starts(df)
            gt_starts = [x["start"] for x in gt_intervals]

            precision, recall, _ = compare_start_times(pred, gt_starts)
            precs.append(precision)
            recs.append(recall)

        except Exception as e:
            print(f"⚠️ Error evaluating {filename}: {e}")

    mean_p = np.mean(precs) if precs else 0
    mean_r = np.mean(recs) if recs else 0
    fbeta = weighted_fbeta(mean_p, mean_r)

    print("\n✅ Test Set Evaluation:")
    print(f"• watt_drop     = {watt_drop}")
    print(f"• Precision     = {mean_p:.4f}")
    print(f"• Recall        = {mean_r:.4f}")
    print(f"• Fβ-Score      = {fbeta:.4f}")


# === Run Evaluation on Test Set ===
evaluate_on_test(watt_drop=11, test_files=test_files)

⚠️ Error evaluating 13363782092_Zwift__Aerobic_Mixup_in_New_York.fit: read_fit_file() takes 1 positional argument but 2 were given
⚠️ Error evaluating i65696340_Zwift__LC16_Lactate_Clearance_Over_Under_in_Watopia.fit: read_fit_file() takes 1 positional argument but 2 were given
⚠️ Error evaluating 13601462878_Zehnbauer.fit: read_fit_file() takes 1 positional argument but 2 were given
⚠️ Error evaluating 12036692734_Exhausted.fit: read_fit_file() takes 1 positional argument but 2 were given
⚠️ Error evaluating 12846436186_Synchronflug.fit: read_fit_file() takes 1 positional argument but 2 were given
⚠️ Error evaluating 11962243206_Wundmanagement.fit: read_fit_file() takes 1 positional argument but 2 were given
⚠️ Error evaluating 13688068283_Luftpresser.fit: read_fit_file() takes 1 positional argument but 2 were given
⚠️ Error evaluating 11783093951_4x2000_sub8.fit: read_fit_file() takes 1 positional argument but 2 were given
⚠️ Error evaluating 13983533934_Technik__30er.fit: read_fit_f

##### Visualization (saving in a folder)

In [15]:
# === Parameters ===
window_size = 5
watt_drop = 11

# === Output folder setup ===
output_dir = "Plots for Final Presentation/Self Model Power"
os.makedirs(output_dir, exist_ok=True)

# === Color Map ===
zone_labels_ordered = ["Zone1", "Zone2", "Zone3", "Zone4", "Zone5", "Zone6", "Zone7"]
base_colors = [
    "#00008B", "#5dade2", "#229954", "#f1c40f",
    "#e67e22", "#e74c3c", "#7b241c"
]
interval_colors = {label: color for label, color in zip(zone_labels_ordered, base_colors)}
interval_colors["Unclassified"] = "gray"
interval_colors["GroundTruth"] = "gray"

# === Convert test_sessions to full path tuples ===
sampled_files = []
for athlete_name, session_name in test_sessions:
    config = athletes.get(athlete_name)
    if not config:
        print(f"⚠️ Athlete '{athlete_name}' not found.")
        continue

    filename = session_name + ".fit"
    allowed = config["allowed_files"]
    if allowed != "all" and filename not in allowed:
        print(f"⚠️ {filename} not allowed for {athlete_name}")
        continue

    sampled_files.append((filename, config["ftp"], config["fit_dir"], config["label_dir"]))

print(f"📂 Visualizing {len(sampled_files)} explicitly selected files.")

# === Process and Visualize ===
for filename, ftp, fit_dir, label_dir in sampled_files:
    print(f"\n🔍 Visualizing: {filename}")
    fit_path = os.path.join(fit_dir, filename)
    label_path = os.path.join(label_dir, filename.replace(".fit", "_with_manual_labels.csv"))

    if not os.path.exists(label_path):
        print(f"⚠️ Skipping {filename} (missing label file)")
        continue

    try:
        df = read_fit_file(fit_path)
        df = prepare_dataframe(df, ftp=ftp, window_size=window_size)
        df = apply_watt_drop_detection(df, watt_drop=watt_drop)

        df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)
        df = group_true_values(df)
        df = enforce_consecutive_intervals(df)
        df = assign_dominant_zone_type_per_interval(df)
        df = merge_consecutive_same_zone_intervals(df)
        df = detect_and_invalidate_stop_resume_events(df)
        df = merge_short_intervals(df)
        df = reassign_first_n_seconds(df, seconds=60)

        pred_intervals = df[df["final_group_start"]][["timestamp", "interval_zone_type"]].copy()
        pred_intervals["end"] = pred_intervals["timestamp"].shift(-1).fillna(df["timestamp"].iloc[-1])

        gt_intervals = load_ground_truth_starts(label_path)

        fig, ax = plt.subplots(figsize=(15, 5))

        for gt in gt_intervals:
            ax.axvspan(gt["start"], gt["end"], color=interval_colors.get(gt["zone_type"], "gray"), alpha=0.2)

        for _, row in pred_intervals.iterrows():
            zone = row["interval_zone_type"]
            color = interval_colors.get(zone, "gray")
            mask = (df["timestamp"] >= row["timestamp"]) & (df["timestamp"] <= row["end"])
            ax.plot(df.loc[mask, "timestamp"], df.loc[mask, "power_roll_avg"], color=color, linewidth=2)

        legend_elements = [Patch(facecolor=interval_colors[z], label=z) for z in zone_labels_ordered]
        legend_elements.append(Patch(facecolor='gray', label='Unclassified'))
        legend_elements.append(Patch(facecolor='gray', alpha=0.2, label='Ground Truth (background)'))

        ax.legend(handles=legend_elements, bbox_to_anchor=(1.01, 1), loc='upper left')
        ax.set_title(f"{filename}\n(watt_drop={watt_drop}, window_size={window_size})")
        ax.set_ylabel("Power (Smoothed)")
        ax.set_xlabel("Time")
        ax.set_xlim(df["timestamp"].iloc[0], df["timestamp"].iloc[-1])
        ax.grid(True)

        # Save to file
        output_filename = filename.replace(".fit", ".png")
        save_path = os.path.join(output_dir, output_filename)
        plt.tight_layout()
        plt.savefig(save_path)
        plt.close(fig)

    except Exception as e:
        print(f"❌ Failed to visualize {filename}: {e}")

📂 Visualizing 11 explicitly selected files.

🔍 Visualizing: 13363782092_Zwift__Aerobic_Mixup_in_New_York.fit

🔍 Visualizing: i65696340_Zwift__LC16_Lactate_Clearance_Over_Under_in_Watopia.fit

🔍 Visualizing: 13601462878_Zehnbauer.fit

🔍 Visualizing: 12036692734_Exhausted.fit

🔍 Visualizing: 12846436186_Synchronflug.fit

🔍 Visualizing: 11962243206_Wundmanagement.fit

🔍 Visualizing: 13688068283_Luftpresser.fit

🔍 Visualizing: 11783093951_4x2000_sub8.fit

🔍 Visualizing: 13983533934_Technik__30er.fit

🔍 Visualizing: 11846980624_Platt_.fit

🔍 Visualizing: 14125110656_Vallah_isch_balla.fit
